In [1]:
!pip3 install beautifulsoup4
!pip3 install requests

In [2]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

In [3]:
!pip3 install beautifulsoup4
!pip3 install requests

# Imports
import requests
from bs4 import BeautifulSoup
import unicodedata
import pandas as pd

# Helper functions
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([bv for i, bv in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    return [i for i in table_cells.strings][0]

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        idx = mass.find("kg")
        new_mass = mass[0:idx+2] if idx != -1 else mass
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    col_name = ' '.join(row.contents).strip()
    if not col_name.isdigit() and col_name:
        return col_name
    return None


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

# Request page
response = requests.get(static_url)
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title.string)

# Extract table headers
html_tables = soup.find_all('table', class_="wikitable")
first_launch_table = html_tables[2]  # third table
column_names = []

for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name:
        column_names.append(name)

print("Column Names:", column_names)

# Create dictionary to hold launch data
launch_dict = {col: [] for col in column_names if col != 'Date and time ( )'}
launch_dict['Version Booster'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

# Extract table rows
for table in soup.find_all('table', "wikitable plainrowheaders collapsible"):
    for rows in table.find_all("tr"):
        if rows.th and rows.th.string and rows.th.string.strip().isdigit():
            flight_number = rows.th.string.strip()
            row = rows.find_all('td')
            
            datatimelist = date_time(row[0])
            launch_dict['Flight No.'].append(flight_number)
            launch_dict['Date'].append(datatimelist[0].strip(','))
            launch_dict['Time'].append(datatimelist[1])
            
            bv = booster_version(row[1])
            if not bv and row[1].a:
                bv = row[1].a.string
            launch_dict['Version Booster'].append(bv)
            
            launch_dict['Launch site'].append(row[2].a.string if row[2].a else None)
            launch_dict['Payload'].append(row[3].a.string if row[3].a else None)
            launch_dict['Payload mass'].append(get_mass(row[4]))
            launch_dict['Orbit'].append(row[5].a.string if row[5].a else None)
            launch_dict['Customer'].append(row[6].a.string if row[6].a else None)
            launch_dict['Launch outcome'].append(list(row[7].strings)[0].strip())
            launch_dict['Booster landing'].append(landing_status(row[8]))

# Create DataFrame
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})

# Save to CSV
df.to_csv('spacex_web_scraped.csv', index=False)
print("Scraping complete, CSV saved!")

List of Falcon 9 and Falcon Heavy launches - Wikipedia
Column Names: ['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


KeyError: 'Booster landing'

In [5]:
import requests
import pandas as pd

# GET request from SpaceX API
url = "https://api.spacexdata.com/v3/launches"
response = requests.get(url)
data = response.json()

# Convert JSON to DataFrame
df = pd.json_normalize(data)

# Check first row of static_fire_date_utc
first_date = df.loc[0, 'static_fire_date_utc']
print("First row static_fire_date_utc:", first_date)

# Extract the year
first_year = pd.to_datetime(first_date).year
print("Year in first row:", first_year)


First row static_fire_date_utc: 2006-03-17T00:00:00.000Z
Year in first row: 2006


In [6]:
df['landing_pad'].isnull().sum()

KeyError: 'landing_pad'

In [7]:
import requests
import pandas as pd

# Fetch launch data from SpaceX API
url = "https://api.spacexdata.com/v3/launches"
response = requests.get(url)
data = response.json()

# Normalize JSON data into a pandas DataFrame
df = pd.json_normalize(data)

# Count missing values in the 'landingPad' column
missing_values_count = df['landing_pad'].isnull().sum()

print(f"Number of missing values in 'landingPad' column: {missing_values_count}")

KeyError: 'landing_pad'